# DPI Demo Notebook
Run inference on a single PDB file or a directory of PDB files to assess the quality of protein-protein interface using deep learning based DPI Model.

IMPORTANT!! Before proceeding, please follow the installation istruction in the README.md

### Create a virtual environment 

In [1]:
# # Create venv and register it as a kernel
# !python -m venv dpi_env
# !dpi_env/bin/pip install ipykernel
# !dpi_env/bin/python -m ipykernel install --user --name=dpi_env --display-name "Python (dpi_env)"

In [2]:
# # Install dpi into the virtual environment
# !git clone https://gitlab.com/ccpem/dpi.git
# !cd dpi && ../dpi_env/bin/pip install -e .

Then go to Kernel → Change Kernel → Python (dpi_env) and all subsequent cells will run inside that environment with no path tricks needed.

## 1. Imports

In [3]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import sys
import time
import json
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from omegaconf import OmegaConf

import torch
from torch.nn import functional as F

from dpi.model import DPIScore
from dpi.grid import Grid
from dpi.processor import PDBProcessor
from dpi.helper import get_files_in_dir

ModuleNotFoundError: No module named 'omegaconf'

## 2. Configuration
Edit the parameters below to match your setup.

In [1]:
# Input/output 
INPUT          = './examples'                    # Path to a PDB file or directory of PDB files
FILE_TYPES     = [".cif", ".pdb"]
OUTPUT_DIR     = './dpiscore_predictions'        # Directory where results will be saved

# Specify Model directory and folde
MODEL_DIR      = './models/dynamicgrids_aug'     # Directory containing model_cfg.yaml and checkpoint(s)
CHECKPOINT     = 'model_k2.pth'                  # Checkpoint name (without .pth), k represent fold

# iAlign parameters
IALIGN_FILE    = None                            # Path to iAlign file (or None to skip)
IALIGN_CUTOFF  = 0.7                             # IS-score cutoff
IRMSD          = 3.0                             # Min iRMSD threshold

# Interface filtering 
MIN_NUM_RESIDUES      = 10
MAX_NEIGHBORS_DIST    = 7.0
MIN_NEIGHBORS_CHAINS  = 30
MAX_NUM_CHAINS        = 30
AUTHOR                = False 

#  Hardware 
GPU = 0                                          # GPU id; set to -1 to force CPU

## 4. Verify Paths and Select Device

In [2]:
if not os.path.exists(MODEL_DIR):
    raise FileNotFoundError(f"Model directory not found: {MODEL_DIR}")

if GPU != -1 and torch.cuda.is_available():
    device = torch.device(f"cuda:{GPU}")
else:
    device = torch.device("cpu")

print(f"Input  : {INPUT}")
print(f"Model  : {MODEL_DIR}")
print(f"Device : {OUTPUT_DIR}")

NameError: name 'os' is not defined

## 5. Prepare PDB Dataset

In [7]:
processor = PDBProcessor(
    min_neighbors_chains=MIN_NEIGHBORS_CHAINS,
    max_neighbors_dist=MAX_NEIGHBORS_DIST,
    min_num_residues=MIN_NUM_RESIDUES,
    max_num_chains=MAX_NUM_CHAINS,
    author=AUTHOR,
    ialign_file_path=IALIGN_FILE,
)
processor.process(INPUT)

Found 2 PDB file(s).


Processing PDBs:   0%|          | 0/2 [00:00<?, ?it/s]

  Processing ./examples/H1157.pdb ...
  Processing ./examples/H1217.pdb ...


Processing PDBs: 100%|██████████| 2/2 [00:00<00:00,  5.21it/s]


14 interface(s) ready for inference.
Done!


## 7. Load Model

In [8]:
model_cfg = OmegaConf.load(f"{MODEL_DIR}/cfg.yaml")

model = DPIScore(model_cfg)
model.load_checkpoint(f"{MODEL_DIR}/{CHECKPOINT}", map_location=device)
model.to(device)
model.eval()

Loading pretrained model from ./models/dynamicgrids_aug/model_k2.pth...
Model loaded successfully.


DPIScore(
  (net): CNN3D(
    (maxpool): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (layer1): SingleConv(
      (single_conv): Sequential(
        (0): Conv3d(5, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
        (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
      )
    )
    (layer2): SingleConv(
      (single_conv): Sequential(
        (0): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
      )
    )
    (layer3): SingleConv(
      (single_conv): Sequential(
        (0): Conv3d(64, 128, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
        (1): BatchNorm3d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
      )
    )
    (layer4): SingleConv(
      (single_conv): Sequential(
  

## 8. Run Inference

In [9]:
predictions = []
start = time.time()

for intf, intf_feats in tqdm(processor.intf_features.items(), desc="Inferring"):
    t0 = time.time()

    dpi_score = model.predict(intf_feats, device)

    print(
        f"Interface: {intf} | "
        f"dpi_score: {min(dpi_score, 0.999):.3f} | "
        f"time: {time.time() - t0:.3f}s"
    )

    intf_meta_info = processor.meta_dict.get(intf, {})
    predictions.append({
        'pdb':       intf_meta_info.get('pdb', None),
        'interface': intf_meta_info.get('interface', None),
        'dpi_score': dpi_score,
    })

print(f"\nInference complete in {time.time() - start:.2f}s")

Inferring:   0%|          | 0/14 [00:00<?, ?it/s]

Inferring:  21%|██▏       | 3/14 [00:00<00:01,  5.94it/s]

Interface: H1157_A_B | dpi_score: 0.999 | time: 0.364s
Interface: H1217_G_C | dpi_score: 0.998 | time: 0.070s
Interface: H1217_G_H | dpi_score: 0.999 | time: 0.129s


Inferring:  36%|███▌      | 5/14 [00:00<00:01,  6.82it/s]

Interface: H1217_C_E | dpi_score: 0.999 | time: 0.060s
Interface: H1217_C_F | dpi_score: 0.999 | time: 0.190s


...

Interface: H1217_I_A | dpi_score: 0.999 | time: 0.134s
Interface: H1217_E_D | dpi_score: 0.999 | time: 0.144s


Inferring:  79%|███████▊  | 11/14 [00:01<00:00,  9.12it/s]

Interface: H1217_A_D | dpi_score: 0.998 | time: 0.078s
Interface: H1217_H_D | dpi_score: 0.998 | time: 0.069s
Interface: H1217_D_F | dpi_score: 0.999 | time: 0.058s


Inferring: 100%|██████████| 14/14 [00:01<00:00,  7.70it/s]

Interface: H1217_J_F | dpi_score: 0.998 | time: 0.137s
Interface: H1217_J_B | dpi_score: 0.999 | time: 0.156s

Inference complete in 1.82s


## 9. Inspect Results

In [10]:
df_results = pd.DataFrame(predictions)
df_results['dpi_score'] = df_results['dpi_score'].clip(upper=0.999)
df_results.reset_index(drop=True, inplace=True)
df_results

,pdb,interface,dpi_score
0,H1157,A_B,0.999000
1,H1217,G_C,0.998094
2,H1217,G_H,0.999000
3,H1217,C_E,0.998664
4,H1217,C_F,0.999000
5,H1217,C_B,0.998168
6,H1217,I_E,0.998983
7,H1217,I_A,0.999000
8,H1217,E_D,0.999000
9,H1217,A_D,0.997517


## 10. Save Results

In [1]:
run_name   = os.path.basename(os.path.splitext(INPUT)[0])
save_dir   = os.path.join(OUTPUT_DIR, run_name)
os.makedirs(save_dir, exist_ok=True)

# Save predictions CSV
csv_path = os.path.join(save_dir, 'inference_results.csv')
df_results.to_csv(csv_path, index=False)
print(f"Predictions saved to: {csv_path}")

# Save meta JSON
meta_path = os.path.join(save_dir, 'meta_labels.json')
with open(meta_path, 'w') as f:
    json.dump(processor.meta_dict, f)
print(f"Meta info saved to  : {meta_path}")

NameError: name 'os' is not defined